## Validação Instalação

In [1]:
import os
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
import importlib.metadata

# =====================================================
# Versões Delta Lake
# =====================================================
print("Versão delta-spark (Python):", importlib.metadata.version("delta-spark"))

# =====================================================
# Configuração de ambiente para Spark + Delta 3.2.1
# =====================================================
os.environ["SPARK_HOME"] = "/usr/local/spark"
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--jars /usr/local/spark/jars/delta-spark_2.12-3.2.1.jar,"
    "/usr/local/spark/jars/delta-storage-3.2.1.jar pyspark-shell"
)

# =====================================================
# Sessão Spark configurada para MinIO + Delta
# =====================================================
spark = (
    SparkSession.builder
    .appName("Delta-Bronze")
    .master("local[*]")
    .config("spark.executor.memory", "1g")
    # Configuração S3A (MinIO)
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    # Integração Delta Lake
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

# =====================================================
# Teste da integração e S3A Handler
# =====================================================
spark._jsc.hadoopConfiguration().set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
print("Handler S3A inicializado com sucesso:", fs)

spark.sql("SELECT version()").show()
#spark.stop()


Versão delta-spark (Python): 3.2.1


26/10/21 13:28:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Handler S3A inicializado com sucesso: org.apache.hadoop.hive.ql.io.ProxyLocalFileSystem@3b63e322
+--------------------+
|           version()|
+--------------------+
|3.5.8 5a48a37b2db...|
+--------------------+

